# Fine-tuning Nemotron 3.5 ASR for Circassian (ady + kbd)

Adapts [`nvidia/nemotron-3.5-asr-streaming-0.6b`](https://huggingface.co/nvidia/nemotron-3.5-asr-streaming-0.6b)
(Cache-Aware FastConformer-RNNT, prompt-conditioned, 40 locales) to **Adyghe (`ady`)** and
**Kabardian (`kbd`)**, reusing the *raw* corpus already compiled for `maq-tts`
(`/mnt/data/raw/` + `manifests/raw.jsonl`, ~377 h: Common Voice + Gukhel + Forvo).

Following the [NVIDIA fine-tuning guide](https://huggingface.co/blog/nvidia/fine-tuning-nemotron-35-asr).
Design decisions (verified against the actual checkpoint internals):

1. **Raw audio + raw casing; typography unified.** Audio stays dirty (`raw_wav`). Transcripts
   keep original casing; only palochka / quote / dash variants are collapsed so the model learns
   one output form. **Cyrillic-only transcripts** — Latin-alphabet Circassian from Common Voice
   (e.g. `Tiêcap'er etajjişeu zetêt.`) is dropped (`maq-tts` rejects these as tokenizer `<unk>`;
   here we gate explicitly because SentencePiece already covers Latin). Misaligned `(audio, text)`
   pairs are dropped via a per-speaker chars/sec MAD gate.
2. **Typography unification only (ported from `maq-tts` `tokenizer.py`).** Collapse **all** palochka
   forms — both code points (`Ӏ` U+04C0, `ӏ` U+04CF) and context-aware stand-ins (Latin `I`/`l`/`1`/`|`,
   Ukrainian `і`/`І`) — to a **single** canonical `Ӏ`, so the same phoneme is never written two ways
   (`Ӏ` is used as the generic palochka in Circassian regardless of case). The stand-in recovery is
   **context-aware** (only fires in Circassian phonotactic context, so Latin words/numbers are left
   alone). Fold quote/dash variants (`«»` → `"`, `–` → `—`, etc.); drop soft hyphens. Casing of every
   **other** letter is preserved. **No** TTS-style casefold, `ё→е`, Latin look-alike folding, or
   stray-sign stripping.
3. **Warm prompt-slot reuse for `ady`/`kbd`.** Language conditioning is a fixed **128-dim one-hot**
   (`prompt_dictionary`). Instead of training cold unused slots, `ady` rides the pretrained
   `uk-UA` slot and `kbd` rides the pretrained `bg-BG` slot. `ru-RU` stays untouched.
4. **Tokenizer coverage.** After typography unification the SentencePiece vocab is still missing
   a handful of pieces (canonical palochka `Ӏ`, em-dash `—`, `: ; ( )`, …). We extend it and
   warm-start the decoder/joint, preserving the original 13,087 rows and keeping `<unk>`'s learned
   emission prior for new rows so high-frequency `Ӏ` does not dominate greedy decode.
5. **Honest evaluation.** A **speaker-disjoint** held-out test set (speakers unseen in training),
   scored at **deployment latency** `att_context_size=[56,0]` (80 ms, 0 ms look-ahead).

Tooling: **uv** (installs), **tqdm** (progress), **wandb** (logging). Run on a GPU box where the
`maq-tts` raw data lives (the `vast.ai` instance from the data-prep notebook).

## 1. Configuration

All knobs in one place. Paths point at the `maq-tts` raw tree (audio stays 22.05 kHz on disk and
is resampled to 16 kHz on the fly by the NeMo/Lhotse dataloader).

In [ ]:
import os
# PyTorch 2.6+ defaults torch.load(weights_only=True); NeMo .nemo archives need full unpickle.
# Must be set before the first torch import anywhere in this kernel (NeMo adapters tutorial pattern).
os.environ.setdefault("TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD", "1")

from pathlib import Path

# --- data (the maq-tts raw tree from the data-prep notebook) ---
DATA_ROOT    = Path("/mnt/data")                       # holds raw/ and manifests/
RAW_MANIFEST = DATA_ROOT / "manifests" / "raw.jsonl"   # {id,lang,speaker,source,text,raw_wav,duration}
OUT          = Path("/mnt/nemo_manifests")             # our NeMo manifests land here
EXP_DIR      = Path("/mnt/exp")                         # checkpoints + final .nemo
NEMO_DIR     = Path("/mnt/NeMo")                        # cloned for the streaming-eval script

# --- model + prompt-slot reuse ---
BASE_MODEL   = "nvidia/nemotron-3.5-asr-streaming-0.6b"
NEW_LANGS    = ("ady", "kbd")
PROMPT_SLOT_FOR_LANG = {"ady": "uk-UA", "kbd": "bg-BG"}  # keep ru-RU untouched; these slots will specialize

# --- data filtering (drop misaligned pairs; typography unified in §3) ---
DUR_MIN, DUR_MAX = 0.5, 20.0          # seconds; model trains on <= 20 s
CPS_MIN, CPS_MAX = 2.0, 32.0          # global chars/sec sanity band (gross-misalignment guard)
MAD_K, MAD_MIN_CLIPS = 3.5, 30        # per-speaker robust chars/sec gate (same as maq-tts)
MIN_ADD_COUNT        = 50             # add a missing char to the tokenizer if it occurs >= this often; rarer -> drop clip
TEST_HOURS_PER_LANG  = 1.0            # speaker-disjoint held-out test, per language

# --- training (fixed step budget, per the blog) ---
MAX_STEPS, WARMUP, VAL_EVERY = 25000, 2500, 2500   # WARMUP=10% of steps (NeMo finetune: 6-10%)
BATCH, LR = 32, 1e-4                  # LR matches NeMo speech_to_text_finetune.yaml default
EVAL_ATT_CONTEXT = "[56,0]"           # 80 ms deployment latency, 0 ms look-ahead

# --- wandb ---
WANDB_PROJECT, WANDB_RUN = "nemotron35-asr-circassian", "ady-kbd-ft"

OUT.mkdir(parents=True, exist_ok=True)
EXP_DIR.mkdir(parents=True, exist_ok=True)
assert RAW_MANIFEST.is_file(), f"raw manifest not found: {RAW_MANIFEST}"
print("raw manifest:", RAW_MANIFEST)

## 2. Install NeMo (uv)

NeMo `26.06+` (from `main`) for the `EncDecRNNTBPEModelWithPrompt` class. `libsndfile`/`ffmpeg`
for audio I/O. Assumes a CUDA PyTorch base image (e.g. the `vast.ai` PyTorch template).

In [ ]:
!apt-get -qq update && apt-get -qq install -y libsndfile1 ffmpeg >/dev/null
!pip install -q uv
!uv pip install --system -q Cython packaging
!uv pip install --system -q "nemo_toolkit[asr] @ git+https://github.com/NVIDIA/NeMo.git@main"
!uv pip install --system -q wandb tqdm jiwer

import wandb; wandb.login()   # paste your key, or set WANDB_API_KEY beforehand

## 3. Build NeMo manifests — unify typography, filter misaligned pairs, carve a speaker-disjoint test set

NeMo wants JSON-lines with `audio_filepath`, `duration`, `text`, plus `target_lang` for prompt
conditioning. From `raw.jsonl` we:

- **unify typography** — context-aware, case-preserving palochka recovery + quote/dash folding
  (the `_PALOCHKA_RE` + `_DIRECT_REWRITES` logic from `maq-tts/src/maq_tts/tokenizer.py`, minus the
  TTS-only casefold / `ё→е` / stray-sign steps);
- drop **Latin-script Circassian** — keep only transcripts that contain Cyrillic after normalization
  (same gate as `maq-tts` `6_build_manifests.py` rejecting tokenizer `<unk>`, but explicit here
  because the base SentencePiece vocab already encodes Latin letters);
- keep **raw audio** (`raw_wav`) and otherwise **raw casing**;
- drop **misaligned** clips: out-of-range duration, a global chars/sec sanity band, and a
  per-speaker robust MAD gate on chars/sec;
- hold out whole **speakers** (unseen in training) per language for an honest test set;
- split the rest into train / a small monitoring val.

In [ ]:
import json, random, re, statistics, unicodedata
from collections import defaultdict
from tqdm.auto import tqdm

# Typography + palochka unification — ported from maq-tts/src/maq_tts/tokenizer.py but made
# CASE-PRESERVING (ASR keeps capitalization). We collapse only visual variants; we do NOT
# casefold, strip stray ъ/ь, fold Latin look-alikes, or map ё→е.
PALOCHKA = "\u04C0"   # single canonical palochka (uppercase Ӏ)

# Quote / dash / misc folds — one canonical glyph each (ASCII '-' stays a word hyphen).
# The lowercase palochka ӏ (U+04CF) is unified to Ӏ as well: in Circassian text palochka is used
# generic regardless of case (here Ӏ×290k vs ӏ×189k), so keeping them distinct would leave
# the SAME phoneme written two ways — exactly the split we must remove so the model outputs one form.
_TRANS = str.maketrans({
    "\u04CF": PALOCHKA, "\u01C0": PALOCHKA,   # lowercase palochka ӏ + latin dental click -> Ӏ
    "\u2013": "\u2014", "\u2012": "\u2014", "\u2015": "\u2014", "\u2212": "\u2014",   # dashes -> em-dash
    "\u00ab": '"', "\u00bb": '"', "\u201c": '"', "\u201d": '"', "\u201e": '"', "\u201f": '"',
    "\u2018": "'", "\u2019": "'", "\u201a": "'", "\u201b": "'",
    "\u2026": "...", "\u00a0": " ", "\u00ad": "",   # ellipsis, nbsp, soft-hyphen
})

# Context-aware palochka recovery (maq-tts _PALOCHKA_RE): a stand-in (1 l i I і І |) becomes the
# palochka ONLY in Circassian phonotactic context -- before a vowel / sentence punctuation, or
# after a palochka-forming consonant -- so Latin words and bare numbers stay intact. Casing of all
# OTHER letters is preserved; only the palochka itself is collapsed to a single uppercase Ӏ.
_PV = "эауыиеоЭАУЫИЕО"
_PC = "клпстфцщчшыиеэаКЛПСТФЦЩЧШЫИЕЭА"
_PS = "1liI\u0456\u0406|"                                  # 1 l i I(latin) і І(cyrillic) |
_PAL_RE = re.compile(rf"(?<!\d)[{_PS}](?=[{_PV}.,!?:;]|$)|(?<=[{_PC}])[{_PS}]")
_WS = re.compile(r"\s+")
# Cyrillic gate — same Unicode block as maq-tts tokenizer._HAS_CYRILLIC_RE. Drops Common Voice
# Latin-alphabet Circassian (Tiêcap'er etajjişeu zetêt.) that maq-tts would reject as <unk>.
_HAS_CYRILLIC_RE = re.compile(r"[\u0400-\u04FF]")

def normalize_asr_text(text: str) -> str:
    text = unicodedata.normalize("NFC", text)
    text = text.replace("<<", '"').replace(">>", '"').translate(_TRANS)
    text = _PAL_RE.sub(PALOCHKA, text)
    return _WS.sub(" ", text).strip()

# --- load raw rows -> NeMo records ---
recs, dropped_non_cyr = [], 0
with open(RAW_MANIFEST, encoding="utf-8") as f:
    for line in tqdm(f, desc="read raw.jsonl"):
        if not line.strip():
            continue
        r = json.loads(line)
        text, dur = normalize_asr_text((r.get("text") or "").strip()), r.get("duration")
        if not text or not dur or r["lang"] not in NEW_LANGS:
            continue
        if not _HAS_CYRILLIC_RE.search(text):
            dropped_non_cyr += 1
            continue
        recs.append({
            "audio_filepath": str((DATA_ROOT / r["raw_wav"]).resolve()),
            "duration": float(dur), "text": text,
            "target_lang": r["lang"], "lang": r["lang"],
            "speaker": f'{r["lang"]}/{r["speaker"]}', "source": r.get("source", ""),
        })
print(f"loaded {len(recs):,} Cyrillic clips ({dropped_non_cyr:,} dropped as Latin-script / non-Cyrillic)")

# --- misalignment filter: duration band + global chars/sec + per-speaker MAD ---
recs = [r for r in recs if DUR_MIN <= r["duration"] <= DUR_MAX]
for r in recs:
    r["cps"] = len(r["text"]) / r["duration"]
recs = [r for r in recs if CPS_MIN <= r["cps"] <= CPS_MAX]

by_spk = defaultdict(list)
for r in recs:
    by_spk[r["speaker"]].append(r)
kept, dropped = [], 0
for rs in by_spk.values():
    cps = [r["cps"] for r in rs]
    if len(rs) >= MAD_MIN_CLIPS:
        med = statistics.median(cps)
        mad = statistics.median([abs(x - med) for x in cps])
        if mad > 0:
            lo, hi = med - MAD_K * 1.4826 * mad, med + MAD_K * 1.4826 * mad
            for r in rs:
                if lo <= r["cps"] <= hi:
                    kept.append(r)
                else:
                    dropped += 1
            continue
    kept.extend(rs)
recs = kept
print(f"after filtering: {len(recs):,} clips ({dropped:,} dropped as per-speaker cps outliers)")

# --- speaker-disjoint held-out test (prefer clean Common Voice read speech) ---
rng = random.Random(13)
held = set()
for lang in NEW_LANGS:
    spk_recs = defaultdict(list)
    for r in recs:
        if r["lang"] == lang:
            spk_recs[r["speaker"]].append(r)
    cands = [s for s, rs in spk_recs.items()
             if any(x["source"] == "common_voice" for x in rs) and 50 <= len(rs) <= 800]
    rng.shuffle(cands)
    acc = 0.0
    for s in cands:
        if acc >= TEST_HOURS_PER_LANG * 3600:
            break
        held.add(s)
        acc += sum(x["duration"] for x in spk_recs[s])

test = [r for r in recs if r["speaker"] in held]
trainval = [r for r in recs if r["speaker"] not in held]
rng.shuffle(trainval)
n_val = max(200, int(0.01 * len(trainval)))
val, train = trainval[:n_val], trainval[n_val:]

# --- write manifests ---
def write(path, rows):
    keys = ("audio_filepath", "duration", "text", "target_lang", "lang", "speaker", "source")
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps({k: r[k] for k in keys}, ensure_ascii=False) + "\n")

write(OUT / "train.json", train)
write(OUT / "val.json", val)
for lang in NEW_LANGS:
    write(OUT / f"test_{lang}.json", [r for r in test if r["lang"] == lang])

hrs = lambda rows: sum(r["duration"] for r in rows) / 3600
print(f"train  {len(train):>7,} clips  {hrs(train):6.1f} h")
print(f"val    {len(val):>7,} clips  {hrs(val):6.1f} h")
for lang in NEW_LANGS:
    t = [r for r in test if r["lang"] == lang]
    spk = sorted({r['speaker'] for r in t})
    print(f"test/{lang} {len(t):>6,} clips  {hrs(t):5.1f} h  speakers={len(spk)} (unseen)")

## 4. Load the model & verify SentencePiece coverage

The NeMo warnings about empty train/val manifests here are expected — dataloaders are wired in §6.

**If `restore_from` fails with `TypeError: '…' object is not callable` after you already ran
`transcribe()` / streaming inference in this kernel:** restart the kernel and re-run from §1
([NeMo #15423](https://github.com/NVIDIA-NeMo/NeMo/issues/15423)). Do **not** monkeypatch
`torch.load` via `torch.serialization.load` — that breaks unpickling on a fresh kernel.

Load the base checkpoint and scan **every character in the training text** (after typography
unification) through the tokenizer. Anything that still encodes to `<unk>` (id 0) is not
representable. After folding `«»`/`–`/`Ӏ`/`ӏ`, we expect a short tail: canonical palochka `Ӏ`,
em-dash `—`, and a few common punct marks (`: ; ( )`). Rare junk is dropped per-clip in §5.

In [ ]:
import os, torch
from collections import Counter
from torch.serialization import load as _torch_load

# NeMo's SaveRestoreConnector passes weights_only=True explicitly (env var alone is not enough).
# Patch only that path — do NOT replace torch.load globally (rebinding via torch.serialization.load
# breaks unpickling on a clean kernel).
PATCH_DIR = EXP_DIR / "_torchpatch"
PATCH_DIR.mkdir(parents=True, exist_ok=True)
(PATCH_DIR / "sitecustomize.py").write_text(
    "import os\n"
    "os.environ.setdefault('TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD', '1')\n"
    "import torch\n"
    "_orig = torch.load\n"
    "def _l(*a, **k):\n"
    "    k['weights_only'] = False\n"
    "    return _orig(*a, **k)\n"
    "torch.load = _l\n"
)

from nemo.core.connectors.save_restore_connector import SaveRestoreConnector

def _nemo_load_state_dict(model_weights, map_location="cpu"):
    return _torch_load(model_weights, map_location=map_location, weights_only=False)

SaveRestoreConnector._load_state_dict_from_disk = staticmethod(_nemo_load_state_dict)

import nemo.collections.asr as nemo_asr
from huggingface_hub import hf_hub_download

local_nemo = hf_hub_download(
    BASE_MODEL, "nemotron-3.5-asr-streaming-0.6b.nemo", force_download=False,
)
model = nemo_asr.models.ASRModel.restore_from(local_nemo, map_location="cpu")
print(type(model).__name__, "| vocab:", model.tokenizer.vocab_size,
      "| languages:", len(set(model.cfg.model_defaults.prompt_dictionary.values())))

sp = model.tokenizer.tokenizer          # underlying SentencePieceProcessor
UNK = sp.unk_id()

char_freq = Counter()
for r in tqdm(train, desc="scan chars"):
    char_freq.update(r["text"])

missing = {ch: n for ch, n in char_freq.items()
           if ch != " " and UNK in sp.encode(ch, out_type=int)}
print(f"\ndistinct chars in train: {len(char_freq)}   |   missing (=> <unk>): {len(missing)}")
for ch, n in sorted(missing.items(), key=lambda x: -x[1]):
    print(f"  {ch!r:>5}  U+{ord(ch):04X}  x{n:,}")
print(f"palochka {PALOCHKA!r}: covered={UNK not in sp.encode(PALOCHKA, out_type=int)}")

## 5. Extend the tokenizer, alias `ady`/`kbd` to warm prompt slots, warm-start the decoder/joint

Two edits to the model, both designed to make the first training run stable:

**(a) Tokenizer.** Append every frequently-occurring piece still missing after typography
unification (canonical palochka `Ӏ`, em-dash `—`, common punct) as `USER_DEFINED` pieces at the
end of the SentencePiece model, preserving the original 13,087 ids. Warm-start the resized
decoder/joint so existing rows survive. **New token rows are cloned from `<unk>` without scaling**
so they inherit `<unk>`'s learned low-emission prior; scaling these rows toward zero made palochka
(`Ӏ`, extremely frequent in Circassian) dominate greedy decode (`ӀӀӀ…`) while loss still dropped.
Clips with rare unrepresentable junk are dropped.

**(b) Language prompt.** Add `ady` and `kbd` as aliases in `prompt_dictionary`, but map them to
trained Cyrillic slots: `ady→uk-UA`, `kbd→bg-BG`. This avoids cold prompt columns and keeps
`ru-RU` untouched.

In [ ]:
import torch
import sentencepiece as spm
from omegaconf import open_dict
from sentencepiece import sentencepiece_model_pb2 as spb

# --- (a1) append frequent missing pieces (canonical palochka, em-dash, common punct, …) ---
add = sorted(ch for ch, n in missing.items() if n >= MIN_ADD_COUNT)
proto = spb.ModelProto()
proto.ParseFromString(sp.serialized_model_proto())   # base tokenizer.model bytes (live, no temp file)
have = {p.piece for p in proto.pieces}
n_added = 0
for ch in add:
    if ch in have:
        continue
    p = proto.pieces.add()
    p.piece, p.score, p.type = ch, 0.0, spb.ModelProto.SentencePiece.USER_DEFINED
    n_added += 1
TOKDIR = Path("/mnt/circ_tokenizer"); TOKDIR.mkdir(parents=True, exist_ok=True)
(TOKDIR / "tokenizer.model").write_bytes(proto.SerializeToString())
(TOKDIR / "vocab.txt").write_text("\n".join(p.piece for p in proto.pieces), encoding="utf-8")
print(f"appended {n_added} pieces {add} -> vocab {len(proto.pieces)}")

# NeMo's own helper verifies the edited proto round-trips through SentencePiece.
sp_check = spm.SentencePieceProcessor()
sp_check.LoadFromSerializedProto(proto.SerializeToString())
for ch in add:
    ids = sp_check.encode(ch, out_type=int)
    assert sp_check.unk_id() not in ids, f"{ch!r} still encodes with <unk>: {ids}"
print("SentencePiece proto OK for added pieces")

# --- (a2) resize via change_vocabulary, then warm-start the preserved rows ---
old_vocab = model.tokenizer.vocab_size
old_dec   = {k: v.clone() for k, v in model.decoder.state_dict().items()}
old_joint = {k: v.clone() for k, v in model.joint.state_dict().items()}
model.change_vocabulary(new_tokenizer_dir=str(TOKDIR), new_tokenizer_type="bpe")
n_added = model.tokenizer.vocab_size - old_vocab
assert n_added > 0, "change_vocabulary did not grow the vocab — check TOKDIR"
print(f"vocab {old_vocab} -> {model.tokenizer.vocab_size} (+{n_added})")

def warm_start_(module, old_sd, n_added):
    """Copy preserved RNNT rows; initialize new token rows from <unk>'s learned prior."""
    new_sd = module.state_dict()
    for k, nt in new_sd.items():
        ot = old_sd[k]
        if ot.shape == nt.shape:
            nt.copy_(ot)
            continue
        diff = [d for d in range(nt.dim()) if nt.shape[d] != ot.shape[d]]
        assert len(diff) == 1 and nt.shape[diff[0]] - ot.shape[diff[0]] == n_added, (k, ot.shape, nt.shape)
        d, v = diff[0], ot.shape[diff[0]]
        idx_old = torch.arange(v)
        idx_new = torch.cat([torch.arange(v - 1), torch.tensor([nt.shape[d] - 1])])
        nt.index_copy_(d, idx_new, ot.index_select(d, idx_old))
        # Rows v-1 .. -2 are the newly added tokens (blank stays at -1).
        # Keep <unk>'s learned emission prior intact; scaling it toward zero made Ӏ dominate greedy decode.
        sl = [slice(None)] * nt.dim()
        sl[d] = slice(v - 1, nt.shape[d] - 1)
        unk = ot.index_select(d, torch.tensor([0]))
        nt[tuple(sl)].copy_(unk.expand_as(nt[tuple(sl)]))
    module.load_state_dict(new_sd)

warm_start_(model.decoder, old_dec, n_added)
warm_start_(model.joint, old_joint, n_added)
print("warm-started decoder + joint | new vocab:", model.tokenizer.vocab_size)

# Full-sentence round-trip — per-char checks in §4 can miss BPE composition bugs.
sp_new = model.tokenizer.tokenizer
unk = sp_new.unk_id()
bad = []
for r in train[:500]:
    ids = sp_new.encode(r["text"], out_type=int)
    if unk in ids:
        bad.append(r["text"][:80])
assert not bad, f"{len(bad)} train clips still contain <unk> after extension, e.g. {bad[:3]}"
print("round-trip token check: 0 UNK in first 500 train clips")

# --- (b) alias ady/kbd to warm, existing Cyrillic prompt slots ---
with open_dict(model.cfg):
    pd = dict(model.cfg.model_defaults.prompt_dictionary)
    slot_ids = {lang: pd[slot] for lang, slot in PROMPT_SLOT_FOR_LANG.items()}
    pd.update(slot_ids)
    model.cfg.model_defaults.prompt_dictionary = pd
print("registered prompt aliases:", {
    lang: (PROMPT_SLOT_FOR_LANG[lang], model.cfg.model_defaults.prompt_dictionary[lang])
    for lang in NEW_LANGS
})

# any char that was <unk> and is NOT Cyrillic stays <unk> (rare non-script junk): drop those clips
leftover = set(missing) - set(add)
if leftover:
    has_junk = lambda t: any(c in leftover for c in t)
    for rows in (train, val, test):
        rows[:] = [r for r in rows if not has_junk(r["text"])]
    write(OUT / "train.json", train)
    write(OUT / "val.json", val)
    for lang in NEW_LANGS:
        write(OUT / f"test_{lang}.json", [r for r in test if r["lang"] == lang])
    print(f"dropped clips containing non-script <unk> chars: {sorted(leftover)}")
else:
    print("all training text is now representable (no leftover <unk>)")

## 6. Fine-tune

Full fine-tune of the streaming RNNT on a fixed step budget (the right way to schedule with
Lhotse iterable data). The Lhotse prompt dataset reads each clip's language from `target_lang`
(`lang_field`) and looks it up in `prompt_dictionary`; `ady` resolves to the pretrained `uk-UA`
prompt slot and `kbd` resolves to the pretrained `bg-BG` prompt slot. Audio is resampled to
16 kHz on the fly. `exp_manager` handles checkpointing/resume (spot-friendly) and wandb logging;
a tqdm bar shows progress.

**If a prior run collapsed to `ӀӀӀ…`:** stop it and delete `/mnt/exp/nemotron35_circassian/`
(or set a fresh `name`) before re-running. Do not resume from that checkpoint.

In [ ]:
import json, torch
from omegaconf import OmegaConf
from nemo.utils.exp_manager import exp_manager

# NeMo `main` subclasses lightning.pytorch.LightningModule; match its namespace for the Trainer
# (mixing `pytorch_lightning` and `lightning.pytorch` triggers PL's mixed-imports TypeError).
try:
    import lightning.pytorch as pl
except ImportError:
    import pytorch_lightning as pl

assert getattr(model, "concat", False), (
    f"prompt conditioning is off (concat={getattr(model, 'concat', None)}); "
    f"got {type(model).__name__} — reload the Nemotron streaming-prompt checkpoint"
)
print("prompt OK:", type(model).__name__, "| concat:", model.concat)

torch.set_float32_matmul_precision("high")   # use Tensor Cores for fp32 matmuls (free speedup)

PD = OmegaConf.to_container(model.cfg.model_defaults.prompt_dictionary, resolve=True)
for lang in NEW_LANGS:
    slot = PROMPT_SLOT_FOR_LANG[lang]
    assert lang in PD, f"{lang} missing from prompt_dictionary"
    assert PD[lang] == PD[slot], f"{lang} should reuse {slot}, got {PD[lang]} vs {PD[slot]}"
print("prompt aliases:", {lang: f"{PROMPT_SLOT_FOR_LANG[lang]}#{PD[lang]}" for lang in NEW_LANGS})
common = dict(
    sample_rate=16000, use_lhotse=True, use_bucketing=False,
    min_duration=DUR_MIN, max_duration=DUR_MAX,
    use_start_end_token=False,
    initialize_prompt_feature=True,
    lang_field="target_lang", text_field="text",
    prompt_dictionary=PD, num_prompts=128, subsampling_factor=8,
    prompt_mode_field="prompt_mode", default_prompt_mode="langID",
)
train_cfg = OmegaConf.create({**common, "manifest_filepath": str(OUT / "train.json"),
                              "shuffle": True, "batch_size": BATCH, "num_workers": 8})
# Per-language val manifests -> NeMo runs one val dataloader per language and logs WER separately
# (keys "ady_val_wer"/"kbd_val_wer"; trailing "_" matches NeMo's name-prefix convention). dl idx 0
# (=ady) also emits the plain "val_wer" that drives checkpointing below.
val_rows = [json.loads(l) for l in open(OUT / "val.json", encoding="utf-8")]
val_manifests, val_names = [], []
for lang in NEW_LANGS:
    write(OUT / f"val_{lang}.json", [r for r in val_rows if r["lang"] == lang])
    val_manifests.append(str(OUT / f"val_{lang}.json")); val_names.append(f"{lang}_")
val_cfg = OmegaConf.create({**common, "manifest_filepath": val_manifests, "name": val_names,
                            "shuffle": False, "batch_size": BATCH, "num_workers": 4})

model.cfg.optim = OmegaConf.create(dict(
    name="adamw", lr=LR, betas=[0.9, 0.98], weight_decay=1e-3,
    sched=dict(name="CosineAnnealing", warmup_steps=WARMUP, min_lr=1e-6, max_steps=MAX_STEPS),
))

class _FreezeEncoder(pl.Callback):
    """Stabilize freshly extended decoder/joint rows before unfreezing the encoder."""
    def __init__(self, until_step=WARMUP):
        self.until_step = until_step
    def on_train_batch_start(self, trainer, pl_module, batch, batch_idx):
        freeze = trainer.global_step < self.until_step
        for p in pl_module.encoder.parameters():
            p.requires_grad = not freeze
        if trainer.global_step == self.until_step:
            print(f"step {trainer.global_step}: encoder unfrozen")

trainer = pl.Trainer(
    devices=1, accelerator="gpu", precision="bf16-mixed",   # multi-GPU: convert to a script + DDP
    max_steps=MAX_STEPS, val_check_interval=VAL_EVERY, limit_val_batches=50,
    num_sanity_val_steps=0, log_every_n_steps=20, gradient_clip_val=1.0,
    enable_checkpointing=False, logger=False,
    callbacks=[_FreezeEncoder()],
)
exp_manager(trainer, OmegaConf.create(dict(
    exp_dir=str(EXP_DIR), name="nemotron35_circassian",
    create_wandb_logger=True,
    wandb_logger_kwargs=dict(
        project=WANDB_PROJECT, name=WANDB_RUN,
        tags=["nemotron-3.5-asr-0.6b", "rnnt-streaming", "full-finetune",
              "prompt-slot-reuse", *NEW_LANGS, f"bs{BATCH}", f"{MAX_STEPS // 1000}k-steps"],
    ),
    create_checkpoint_callback=True, resume_if_exists=False, resume_ignore_no_checkpoint=True,
    checkpoint_callback_params=dict(monitor="val_wer", mode="min", save_top_k=2, always_save_nemo=True),
)))

model.set_trainer(trainer)
model.setup_training_data(train_cfg)
model.setup_multiple_validation_data(val_cfg)
trainer.fit(model)

FT_NEMO = EXP_DIR / "nemotron35_circassian.nemo"
model.save_to(str(FT_NEMO))
print("saved fine-tuned model ->", FT_NEMO)

## 7. Evaluate WER at deployment latency (`att_context_size=[56,0]`, 80 ms)

The blog's key discipline: **evaluate on held-out data at the latency you'll deploy.** We run
NeMo's cache-aware *streaming* inference (true chunked streaming with caches, 0 ms look-ahead) on
the speaker-disjoint test set, with `target_lang=ady`/`kbd`; those aliases resolve to the reused
`uk-UA`/`bg-BG` prompt slots saved in the fine-tuned checkpoint. The script prints the streaming
WER directly. This is raw WER (no reference normalization), so it is a conservative, honest number.

In [ ]:
import os, re, subprocess

if not NEMO_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/NVIDIA/NeMo.git", str(NEMO_DIR)], check=True)
SCRIPT = NEMO_DIR / "examples/asr/asr_cache_aware_streaming/speech_to_text_cache_aware_streaming_infer.py"

def streaming_wer(nemo_path, lang, batch_size=BATCH):
    """Cache-aware streaming eval on GPU; halve batch_size and retry on CUDA OOM."""
    while True:
        out = subprocess.run(
            ["python", str(SCRIPT),
             f"model_path={nemo_path}",
             f"dataset_manifest={OUT / f'test_{lang}.json'}",
             f"att_context_size={EVAL_ATT_CONTEXT}",
             f"target_lang={lang}",
             "strip_lang_tags=true",
             "cuda=0",                       # force GPU (None would silently fall back to CPU)
             f"batch_size={batch_size}",
             f"output_path={EXP_DIR / f'pred_{lang}.json'}"],
            capture_output=True, text=True,
            env={
                **os.environ,
                "TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD": "1",
                "PYTHONPATH": f"{PATCH_DIR}{os.pathsep}{os.environ.get('PYTHONPATH', '')}",
            })
        log = out.stdout + out.stderr
        if "out of memory" in log.lower() and batch_size > 1:
            batch_size //= 2
            print(f"[{lang}] CUDA OOM -> retrying at batch_size={batch_size}")
            continue
        m = re.search(r"WER% of streaming mode:\s*([\d.]+)", log)
        if m is None:
            print(out.stdout[-2000:]); print(out.stderr[-2000:])
        return float(m.group(1)) if m else None

for lang in NEW_LANGS:
    wer = streaming_wer(FT_NEMO, lang)
    print(f"[{lang}] fine-tuned streaming WER @ {EVAL_ATT_CONTEXT} (80 ms): {wer}%")